# Trabalho IAA011 – Visão Computacional

## Exercício 2: Redes Neurais Convolucionais (Transfer Learning)

Utilizamos as mesmas bases de Treino e Teste do exercício 1. As redes VGG16 e ResNet50 são usadas com
pesos pré-treinados (ImageNet). As camadas convolucionais ficam congeladas e novas camadas *Fully
Connected* são criadas e treinadas para o problema de 4 classes (0, 1+, 2+, 3+). Comparamos os treinos
de 10 épocas com e sem *Data Augmentation*, para as duas arquiteturas.

### Passo 1 – Importações e configurações iniciais

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet50
from tensorflow.keras.models import Model

### Configuração de pastas e parâmetros

Execute as células em ordem, com a pasta de trabalho do kernel definida como **Trabalho Imagens - IAA**,
onde estão este notebook e as pastas `Train_Warwick` e `Test_Warwick` (mesma pasta usada no exercício 1).

In [2]:
# Execute o notebook com a pasta de trabalho "Trabalho Imagens - IAA".
BASE_DIR = Path.cwd()
TRAIN_DIR = BASE_DIR / "Train_Warwick" / "Train_4cls_amostra"
TEST_DIR = BASE_DIR / "Test_Warwick" / "Test_4cl_amostra"
OUTPUT_DIR = BASE_DIR / "saidas_exercicio2"

print("Pasta de execução:", BASE_DIR)
for pasta in (TRAIN_DIR, TEST_DIR):
    if not pasta.is_dir():
        raise FileNotFoundError(
            f"Pasta não encontrada: {pasta}. "
            "Configure a pasta de trabalho do kernel como 'Trabalho Imagens - IAA'."
        )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10
CLASSES = ["0", "1", "2", "3"]
NOMES_CLASSES = ["0", "1+", "2+", "3+"]

print("Pasta treino existe:", TRAIN_DIR.exists(), "->", TRAIN_DIR)
print("Pasta teste existe:", TEST_DIR.exists(), "->", TEST_DIR)

Pasta de execução: C:\Users\deuss\Documents\IAA\IAA004\iaa002-trabalho-python\TRABALHO_DE_IAA011\Trabalho Imagens - IAA
Pasta treino existe: True -> C:\Users\deuss\Documents\IAA\IAA004\iaa002-trabalho-python\TRABALHO_DE_IAA011\Trabalho Imagens - IAA\Train_Warwick\Train_4cls_amostra
Pasta teste existe: True -> C:\Users\deuss\Documents\IAA\IAA004\iaa002-trabalho-python\TRABALHO_DE_IAA011\Trabalho Imagens - IAA\Test_Warwick\Test_4cl_amostra


### Passo 2 – Carregar as imagens e reproduzir a divisão do exercício 1

Usamos a mesma função de manifesto e a mesma divisão por paciente (`StratifiedGroupKFold` com
`random_state=42`) do exercício 1, para que treino e validação sejam idênticos aos já usados na
extração de características.

In [3]:
def obter_id_paciente(nome_arquivo):
    partes = Path(nome_arquivo).stem.split("_")
    if len(partes) < 3:
        raise ValueError(f"Nome de arquivo fora do padrão esperado: {nome_arquivo}")
    return partes[0]


def carregar_manifesto(pasta_base):
    if not pasta_base.is_dir():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta_base.resolve()}")

    linhas = []
    for pasta_classe in sorted(p for p in pasta_base.iterdir() if p.is_dir()):
        for arquivo in sorted(pasta_classe.rglob("*")):
            if arquivo.is_file() and arquivo.suffix.lower() in IMAGE_EXTENSIONS:
                linhas.append({
                    "image_path": str(arquivo.resolve()),
                    "filename": arquivo.name,
                    "patient_id": obter_id_paciente(arquivo.name),
                    "label": pasta_classe.name,
                })

    if not linhas:
        raise ValueError(f"Nenhuma imagem encontrada em {pasta_base}")
    return pd.DataFrame(linhas)


df_treino_original = carregar_manifesto(TRAIN_DIR)
df_teste = carregar_manifesto(TEST_DIR)

divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
indices_treino, indices_validacao = next(
    divisor.split(
        df_treino_original,
        df_treino_original["label"],
        groups=df_treino_original["patient_id"],
    )
)

df_treino = df_treino_original.iloc[indices_treino].copy()
df_validacao = df_treino_original.iloc[indices_validacao].copy()

assert set(df_treino["patient_id"]).isdisjoint(set(df_validacao["patient_id"]))
assert set(df_treino["patient_id"]).isdisjoint(set(df_teste["patient_id"]))
assert set(df_validacao["patient_id"]).isdisjoint(set(df_teste["patient_id"]))

print("Imagens de treino:", len(df_treino))
print("Imagens de validação:", len(df_validacao))
print("Imagens de teste:", len(df_teste))

Imagens de treino: 476
Imagens de validação: 117
Imagens de teste: 371


### Passo 3 – Codificar os rótulos

As classes textuais `0, 1+, 2+, 3+` são convertidas para os índices inteiros `0, 1, 2, 3`, na mesma
ordem usada no exercício 1.

In [4]:
LABEL_TO_IDX = {rotulo: indice for indice, rotulo in enumerate(CLASSES)}

for df in (df_treino, df_validacao, df_teste):
    df["label_idx"] = df["label"].map(LABEL_TO_IDX)

print(df_treino["label_idx"].value_counts().sort_index())

label_idx
0    116
1    120
2    120
3    120
Name: count, dtype: int64


### Passo 4 – Pipeline `tf.data`: redimensionamento e *Data Augmentation*

As imagens originais têm 250x250 pixels; são redimensionadas para 224x224 (entrada esperada pela VGG16 e
pela ResNet50). O *Data Augmentation* é leve e evita alterar as cores: apenas espelhamento horizontal,
pequena rotação, translação e zoom. O pré-processamento específico de cada arquitetura (normalização de
pixels) é aplicado depois do redimensionamento/augmentation.

In [5]:
camadas_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.08),
], name="data_augmentation")


def carregar_imagem(caminho):
    imagem = tf.io.read_file(caminho)
    imagem = tf.io.decode_png(imagem, channels=3)
    imagem = tf.image.resize(imagem, IMG_SIZE)
    return imagem


def construir_dataset(df, preprocess_fn, augment=False, shuffle=False, batch_size=BATCH_SIZE):
    caminhos = tf.constant(df["image_path"].to_numpy())
    rotulos = tf.constant(df["label_idx"].to_numpy(), dtype=tf.int32)

    dataset = tf.data.Dataset.from_tensor_slices((caminhos, rotulos))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df), seed=42)

    dataset = dataset.map(
        lambda caminho, rotulo: (carregar_imagem(caminho), rotulo),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    dataset = dataset.batch(batch_size)

    if augment:
        dataset = dataset.map(
            lambda imagens, rotulos: (camadas_augmentation(imagens, training=True), rotulos),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    dataset = dataset.map(
        lambda imagens, rotulos: (preprocess_fn(imagens), rotulos),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return dataset.prefetch(tf.data.AUTOTUNE)

### Passo 5 – Construir o modelo (base congelada + novas camadas *Fully Connected*)

A camada convolucional pré-treinada (VGG16 ou ResNet50, `include_top=False`) fica congelada
(`trainable = False`). Sobre ela, criamos um `GlobalAveragePooling2D` seguido de uma camada densa e da
camada de saída softmax com 4 classes. Apenas essas novas camadas são treinadas.

In [6]:
BASES = {
    "VGG16": (VGG16, preprocess_vgg16),
    "ResNet50": (ResNet50, preprocess_resnet50),
}


def construir_modelo(nome_base):
    classe_base, preprocess_fn = BASES[nome_base]
    base = classe_base(weights="imagenet", include_top=False, input_shape=IMG_SIZE + (3,))
    base.trainable = False

    entrada = base.input
    saida = layers.GlobalAveragePooling2D()(base.output)
    saida = layers.Dense(256, activation="relu")(saida)
    saida = layers.Dropout(0.3)(saida)
    saida = layers.Dense(len(CLASSES), activation="softmax")(saida)

    modelo = Model(inputs=entrada, outputs=saida, name=f"{nome_base}_transfer")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo, preprocess_fn

### Passo 6 – Treinar VGG16 e ResNet50, com e sem *Data Augmentation*

Para cada arquitetura treinamos duas configurações durante 10 épocas: sem *Data Augmentation* e com
*Data Augmentation*. O histórico de treino (perda e acurácia por época) é salvo em CSV.

In [7]:
modelos_treinados = {}
historicos = []

for nome_base in BASES:
    for usar_augmentation in (False, True):
        sufixo = "com_augmentation" if usar_augmentation else "sem_augmentation"
        nome_config = f"{nome_base}_{sufixo}"
        print(f"\n=== Treinando {nome_config} ===", flush=True)

        modelo, preprocess_fn = construir_modelo(nome_base)

        ds_treino = construir_dataset(
            df_treino, preprocess_fn, augment=usar_augmentation, shuffle=True
        )
        ds_validacao = construir_dataset(df_validacao, preprocess_fn, augment=False, shuffle=False)

        historico = modelo.fit(
            ds_treino,
            validation_data=ds_validacao,
            epochs=EPOCHS,
            verbose=2,
        )

        df_historico = pd.DataFrame(historico.history)
        df_historico.insert(0, "epoca", range(1, EPOCHS + 1))
        df_historico.insert(0, "configuracao", nome_config)
        df_historico.to_csv(OUTPUT_DIR / f"historico_{nome_config}.csv", index=False)
        historicos.append(df_historico)

        modelos_treinados[nome_config] = (modelo, preprocess_fn)

historico_completo = pd.concat(historicos, ignore_index=True)
historico_completo.to_csv(OUTPUT_DIR / "historico_todos.csv", index=False)
display(historico_completo)


=== Treinando VGG16_sem_augmentation ===


       0/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   16384/58889256 ━━━━━━━━━━━━━━━━━━━━ 3:05 3us/step

   49152/58889256 ━━━━━━━━━━━━━━━━━━━━ 2:41 3us/step

   81920/58889256 ━━━━━━━━━━━━━━━━━━━━ 2:58 3us/step

  147456/58889256 ━━━━━━━━━━━━━━━━━━━━ 2:01 2us/step

... (384 outputs left out)

30/30 - 13s - 434ms/step - accuracy: 0.9874 - loss: 0.0395 - val_accuracy: 0.9487 - val_loss: 0.1416


Epoch 10/10


30/30 - 13s - 444ms/step - accuracy: 0.9853 - loss: 0.0568 - val_accuracy: 0.9402 - val_loss: 0.2190


,configuracao,epoca,accuracy,loss,val_accuracy,val_loss
0,VGG16_sem_augmentation,1,0.689076,1.213657,0.735043,0.778848
1,VGG16_sem_augmentation,2,0.903361,0.240937,0.743590,0.779034
2,VGG16_sem_augmentation,3,0.939076,0.158754,0.786325,0.642707
3,VGG16_sem_augmentation,4,0.976891,0.078337,0.829060,0.687498
4,VGG16_sem_augmentation,5,0.966387,0.084944,0.811966,0.713029
5,VGG16_sem_augmentation,6,0.987395,0.042646,0.786325,0.895379
6,VGG16_sem_augmentation,7,0.991597,0.031782,0.820513,0.774098
7,VGG16_sem_augmentation,8,0.993697,0.028565,0.837607,0.637325
8,VGG16_sem_augmentation,9,0.991597,0.018657,0.820513,0.726379
9,VGG16_sem_augmentation,10,0.997899,0.017975,0.820513,0.687835


### Passo 7 – Calcular as métricas pela matriz de confusão

Mesma definição usada no exercício 1: Sensibilidade = TP / (TP + FN), Especificidade = TN / (TN + FP) e
F1 = 2TP / (2TP + FP + FN), calculadas por classe e resumidas pela média macro.

In [8]:
def calcular_metricas(matriz):
    linhas = []
    for i, classe in enumerate(NOMES_CLASSES):
        tp = matriz[i, i]
        fn = matriz[i, :].sum() - tp
        fp = matriz[:, i].sum() - tp
        tn = matriz.sum() - tp - fn - fp

        sensibilidade = tp / (tp + fn) if tp + fn > 0 else 0
        especificidade = tn / (tn + fp) if tn + fp > 0 else 0
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 0

        linhas.append({
            "classe": classe,
            "sensibilidade": sensibilidade,
            "especificidade": especificidade,
            "f1": f1,
        })
    return pd.DataFrame(linhas)

### Passo 8 – Aplicar os modelos treinados na base de Teste

Para cada configuração (arquitetura x com/sem *Data Augmentation*), geramos as predições na base de
teste, a matriz de confusão e as métricas por classe. As tabelas são salvas em CSV para a entrega.

In [9]:
RESULTADOS_DIR = OUTPUT_DIR / "resultados"
RESULTADOS_DIR.mkdir(exist_ok=True)

resultados = []

for nome_config, (modelo, preprocess_fn) in modelos_treinados.items():
    ds_teste = construir_dataset(df_teste, preprocess_fn, augment=False, shuffle=False)

    probabilidades = modelo.predict(ds_teste, verbose=0)
    predicoes_idx = probabilidades.argmax(axis=1)
    predicoes = [CLASSES[i] for i in predicoes_idx]
    y_teste = df_teste["label"].to_numpy()

    matriz = confusion_matrix(y_teste, predicoes, labels=CLASSES)
    tabela_matriz = pd.DataFrame(
        matriz, index=NOMES_CLASSES, columns=NOMES_CLASSES
    ).rename_axis(index="Real", columns="Predita")
    metricas = calcular_metricas(matriz)

    print(f"\n=== {nome_config} ===")
    print("Matriz de confusão:")
    display(tabela_matriz)
    print("Métricas por classe:")
    display(metricas.round(4))

    tabela_matriz.to_csv(RESULTADOS_DIR / f"matriz_{nome_config}.csv")
    metricas.to_csv(RESULTADOS_DIR / f"metricas_{nome_config}.csv", index=False)
    pd.DataFrame({
        "classe_real": y_teste, "classe_predita": predicoes
    }).to_csv(RESULTADOS_DIR / f"predicoes_{nome_config}.csv", index=False)

    resultados.append({
        "configuracao": nome_config,
        "sensibilidade_macro": metricas["sensibilidade"].mean(),
        "especificidade_macro": metricas["especificidade"].mean(),
        "f1_macro": metricas["f1"].mean(),
    })

resultados_df = pd.DataFrame(resultados)
display(resultados_df.round(4))


=== VGG16_sem_augmentation ===
Matriz de confusão:


Predita,0,1+,2+,3+
Real,,,,
0,94,6,1,0
1+,6,43,41,0
2+,0,14,66,10
3+,0,0,0,90


Métricas por classe:


,classe,sensibilidade,especificidade,f1
0,0,0.9307,0.9778,0.9353
1,1+,0.4778,0.9288,0.5621
2,2+,0.7333,0.8505,0.6667
3,3+,1.0000,0.9644,0.9474



=== VGG16_com_augmentation ===
Matriz de confusão:


Predita,0,1+,2+,3+
Real,,,,
0,94,6,1,0
1+,13,32,45,0
2+,0,11,78,1
3+,0,0,0,90


... (7 outputs left out)

Predita,0,1+,2+,3+
Real,,,,
0,99,2,0,0
1+,2,78,10,0
2+,0,10,72,8
3+,0,0,0,90


Métricas por classe:


,classe,sensibilidade,especificidade,f1
0,0,0.9802,0.9926,0.9802
1,1+,0.8667,0.9573,0.8667
2,2+,0.8000,0.9644,0.8372
3,3+,1.0000,0.9715,0.9574


,configuracao,sensibilidade_macro,especificidade_macro,f1_macro
0,VGG16_sem_augmentation,0.7855,0.9304,0.7779
1,VGG16_com_augmentation,0.7882,0.9310,0.7719
2,ResNet50_sem_augmentation,0.8947,0.9661,0.8932
3,ResNet50_com_augmentation,0.9117,0.9715,0.9104


### Passo 9 – Comparar e indicar o melhor resultado

Como no exercício 1, usamos **F1 macro** como métrica principal, por dar o mesmo peso às quatro classes
e considerar falsos positivos e falsos negativos simultaneamente. A tabela é ordenada por essa métrica.

In [10]:
comparacao = resultados_df.sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(comparacao.round(4))
comparacao.to_csv(RESULTADOS_DIR / "comparacao_modelos.csv", index=False)

melhor_f1 = comparacao["f1_macro"].max()
melhores = comparacao[comparacao["f1_macro"] == melhor_f1]

print("Melhor resultado pelo F1 macro:")
for _, resultado in melhores.iterrows():
    print(f"{resultado['configuracao']}: {resultado['f1_macro']:.4f}")
print("Justificativa: maior F1 macro, atribuindo o mesmo peso às quatro classes.")
print("CSVs salvos em:", RESULTADOS_DIR)

,configuracao,sensibilidade_macro,especificidade_macro,f1_macro
0,ResNet50_com_augmentation,0.9117,0.9715,0.9104
1,ResNet50_sem_augmentation,0.8947,0.9661,0.8932
2,VGG16_sem_augmentation,0.7855,0.9304,0.7779
3,VGG16_com_augmentation,0.7882,0.9310,0.7719


Melhor resultado pelo F1 macro:
ResNet50_com_augmentation: 0.9104
Justificativa: maior F1 macro, atribuindo o mesmo peso às quatro classes.
CSVs salvos em: C:\Users\deuss\Documents\IAA\IAA004\iaa002-trabalho-python\TRABALHO_DE_IAA011\Trabalho Imagens - IAA\saidas_exercicio2\resultados
